# Source-Aware 中文词句分类器 — Colab 训练 notebook

本 notebook 在 Google Colab (GPU) 上完成全流程：
1. 环境准备
2. 上传本地项目 / 克隆
3. 下载原始词典数据
4. 构建 source-aware JSONL 数据集（弱监督）
5. 训练 Tfidf + LogisticRegression baseline
6. 训练 BERT-base-chinese 多标签分类器
7. 预测示例 + 对比

**运行前**：`Runtime → Change runtime type → GPU`

## 0. 环境与依赖

In [ ]:
!pip -q install scikit-learn joblib numpy
!pip -q install torch transformers

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 1. 获取项目代码

**方式 A（推荐）**：直接从 GitHub 克隆。

In [ ]:
# 方式 A: git clone (推荐 — 代码总是最新)
import os
if not os.path.exists('zh-classifier-pipeline'):
    !git clone https://github.com/moyx782/zh-classifier-pipeline.git
os.chdir('/content/zh-classifier-pipeline')
print('当前目录:', os.getcwd())
print('目录内容:', os.listdir('.'))

In [ ]:
# 方式 B (备选): 上传 project.zip 手动解压
# 如果你修改了代码想测试，先在本地打包上传 project.zip
# import zipfile
# if os.path.exists('/content/project.zip'):
#     with zipfile.ZipFile('/content/project.zip','r') as z:
#         z.extractall('/content/zh-classifier-pipeline')
#     print('project.zip 已解压覆盖')
# os.chdir('/content/zh-classifier-pipeline')
# print('目录内容:', os.listdir('.'))

In [ ]:
import os, zipfile

if os.path.exists('project.zip'):
    with zipfile.ZipFile('project.zip', 'r') as z:
        z.extractall('.')
    print('project.zip 已解压')
else:
    print('没有 project.zip，请把 src/ outputs/ 等目录手动上传到 /content/')

os.chdir('/content')
print('当前目录:', os.getcwd())
print('目录内容:', os.listdir('.'))

## 2. 下载原始词典数据（GitHub 主源）

约 60MB，几秒完成。

In [ ]:
!python src/fetch_sources.py

## 3. 构建 source-aware JSONL 数据集

35 万条样本，弱监督自动打标，1-2 分钟。

In [ ]:
!python src/build_dataset.py

In [ ]:
# 查看构建报告
import json
with open('outputs/build_report.json', encoding='utf-8') as f:
    rpt = json.load(f)
print('总样本:', rpt['total_samples'])
print('needs_review:', rpt['needs_review_count'])
print('confidence:', rpt['confidence_stats'])
print('\ncognitive 分布:')
for k, v in sorted(rpt['cognitive_label_counts'].items(), key=lambda x: -x[1]):
    print(f'  {k}: {v}')

## 4. 训练 Tfidf + LR baseline（轻量）

CPU 即可，2 分钟。

In [ ]:
!python src/train_baseline.py

In [ ]:
# baseline 预测示例
examples = ['接口调用失败', 'Agent 调用工具失败', '机器人路径规划']
for t in examples:
    print(f'\n--- {t} ---')
    !python src/predict.py --text "$t"

## 5. 训练 BERT-base-chinese 多标签分类器

Colab T4 GPU，5 万样本 3 epochs，约 10-15 分钟。

选项：
- `--max-samples 50000` 快速实验
- `--min-confidence 0.72` 只用高置信样本（更干净但更少）
- 全量 35 万样本需要 1-2 小时

In [ ]:
# 快速实验：5 万样本，3 epochs
!python src/train_transformer.py --model bert-base-chinese --max-samples 50000 --epochs 3 --batch-size 64 --lr 2e-5

### (可选) 高置信子集训练

用 `--min-confidence 0.72` 过滤掉 34 万条噪声，只训高置信的 ~1.6 万条。标签更干净。

In [ ]:
# # 可选: 高置信子集
# !python src/train_transformer.py --min-confidence 0.72 --epochs 5 --batch-size 32

### (可选) 全量训练

35 万样本，3 epochs，T4 GPU 约 1 小时。

In [ ]:
# # 可选: 全量训练
# !python src/train_transformer.py --epochs 3 --batch-size 64

## 6. BERT 预测 + 与 baseline 对比

In [ ]:
examples = ['接口调用失败', 'Agent 调用工具失败', '机器人路径规划', '数据库连接超时', '语义表征定义']
for t in examples:
    print(f'\n===== {t} =====')
    print('--- Tfidf baseline ---')
    !python src/predict.py --text "$t" --json
    print('--- BERT ---')
    !python src/predict_transformer.py --text "$t"

## 7. 下载模型到本地

把训练好的 BERT 模型打包下载回本地。

In [ ]:
import shutil

# 打包 BERT best 模型
if os.path.exists('outputs/models/source_aware_transformer/best'):
    shutil.make_archive('source_aware_transformer_best', 'zip', 'outputs/models/source_aware_transformer/best')
    print('BERT best 模型已打包: source_aware_transformer_best.zip')
    print('点左侧 Files 面板 → 右键下载')
else:
    print('BERT 模型目录不存在，请先跑 step 5')

## 8. 可视化标签分布

(可选) 查看数据集标签分布、confidence 直方图。

In [ ]:
import json
import matplotlib.pyplot as plt
from collections import Counter

samples = []
with open('outputs/universal_zh_classifier.source_aware.samples.jsonl', encoding='utf-8') as f:
    for line in f:
        samples.append(json.loads(line))

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Cognitive labels
cog = Counter()
for s in samples:
    for lb in s['cognitive_labels']:
        cog[lb] += 1
axes[0].barh(list(cog.keys()), list(cog.values()))
axes[0].set_title('Cognitive Labels')
axes[0].invert_yaxis()

# Domain labels
dom = Counter()
for s in samples:
    for lb in s['domain_labels']:
        dom[lb] += 1
axes[1].barh(list(dom.keys()), list(dom.values()))
axes[1].set_title('Domain Labels')
axes[1].invert_yaxis()

# Confidence histogram
confs = [s['confidence'] for s in samples]
axes[2].hist(confs, bins=30)
axes[2].set_title('Confidence Distribution')
axes[2].axvline(0.72, color='red', linestyle='--', label='needs_review threshold')
axes[2].legend()

plt.tight_layout()
plt.show()

## 备注

- **不是金标**：所有标签来自弱监督（关键词规则），`needs_review=true` 占多数，适合冷启动+人工审核。
- **每条可追溯**：`source_repo / source_record_id / raw_fields / license_hint` 全保留。
- **BERT 首次下载** `bert-base-chinese` 约 400MB，Colab 网络快。
- 想提升效果：增强 `src/label_rules.py` 关键词 → `build_dataset.py` → 重新训练。